# 02c. DQ Annotation Layer (v3)

Вход: `european_flights_enriched_v4.parquet` (149M строк, 29 788 рейсов, 33 колонки).
Выход: `european_flights_annotated_v3.parquet` (149M строк, 41 колонка).

## Что делает 02c

02c это последний слой DQ перед моделями. Он не модифицирует существующие данные, а аннотирует каждую точку метками качества. Методологический сдвиг от старой v2 (которая называлась `02c_dq_filter`): мы не отбрасываем точки, а помечаем их интерпретируемыми флагами, чтобы downstream-слои сами решали, как с ними работать.

## Три категории флагов

После diagnostic-анализа enriched_v4 выявлены три независимые категории проблемных точек, требующие разной обработки:

1. `dq_hard_flag`: raw телеметрия сломана. Точка непригодна для физического анализа. Включает stale_altitude, threshold_bad из 02_preprocessing_v3 и новый gap_flag.
2. `dq_soft_flag`: derived dynamics подозрительны, но raw data может быть OK. Включает spike_corrected из 02_preprocessing_v3 и новый dq_derivative_bad (физически невозможные acceleration/turn_rate/vertical_accel/energy_rate из wide differencing artefacts).
3. `feature_quality_flag`: feature engineering artefact, raw data полностью валидна. Включает phase_inconsistent (phase classifier absorption через smooth_phases) и energy_deviation_extreme.

## Что НЕ делает 02c

- Не удаляет строки.
- Не модифицирует существующие колонки enriched_v4.
- Не считает anomaly scores (это 03_models).
- Не строит модели.
- Не делает train/test split.
- Не считает sliding window context (`gap_context`, `dq_context`): 03_models построит свои windows и проверит их состав сам.

## Pipeline

1. `02_preprocessing_v3` (clean_v3, 21 col).
2. `02b_feature_engineering_v4` (enriched_v4, 33 col).
3. `02c_dq_annotation_v3` (annotated_v3, 41 col). Это текущий ноутбук.
4. `03_models` (использует MODEL_FEATURES_V3 = 12 признаков плюс DQ contracts).

## 1. Setup

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import os
import gc
import time
import glob
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = '/content/drive/MyDrive/thesis_processed'
OUTPUT_DIR = DATA_DIR

INPUT_PATH = os.path.join(DATA_DIR, 'european_flights_enriched_v4.parquet')
OUTPUT_PATH = os.path.join(OUTPUT_DIR, 'european_flights_annotated_v3.parquet')

print(f'Input:  {INPUT_PATH}')
print(f'Output: {OUTPUT_PATH}')
print(f'Input exists: {os.path.exists(INPUT_PATH)}')
print(f'Input size: {os.path.getsize(INPUT_PATH) / 1e9:.2f} GB')

# Существующие DQ-флаги из 02_preprocessing_v3
DQ_FLAGS_PREPROCESS = [
    'altitude_threshold_bad',
    'groundspeed_threshold_bad',
    'vertical_rate_threshold_bad',
    'altitude_spike_corrected',
    'groundspeed_spike_corrected',
    'vertical_rate_spike_corrected',
    'stale_altitude',
]

# Phase code в name
PHASE_NAMES = {-1: 'unknown', 0: 'ground', 1: 'takeoff', 2: 'climb',
               3: 'cruise', 4: 'descent', 5: 'approach', 6: 'landing'}

# Воспроизводимость
RANDOM_STATE = 1321

print('Setup complete.')

Mounted at /content/drive
Input:  /content/drive/MyDrive/thesis_processed/european_flights_enriched_v4.parquet
Output: /content/drive/MyDrive/thesis_processed/european_flights_annotated_v3.parquet
Input exists: True
Input size: 8.36 GB
Setup complete.


## 2. Annotation thresholds

Все пороги объявлены явно, они должны быть видны на ревью и в дипломном тексте.

### Temporal gap

`gap_flag` помечает точки после временного разрыва > 5 секунд. Sampling rate ADS-B в датасете около 1 Hz (P95 dt_sec ≈ 1.0s, P99.9 ≈ 2s), поэтому всё выше 2s уже необычно. Threshold 5s сбалансированный: ловит реальные out-of-coverage gaps, не реагирует на мелкие reception glitches.

### Derivative DQ

Физически невозможные значения производных кинематики. Wide differencing (N=5) на границах рейсов или вокруг NaN-точек может дать численные артефакты. Эти пороги далеко выше любых реальных манёвров коммерческой авиации:

- acceleration: 15 kts/s ≈ 28 km/h за секунду (4x выше typical max take-off acceleration ~3.5 kts/s).
- turn_rate: 10 deg/s ≈ 60 deg/6s (5x выше standard rate turn 3 deg/s).
- vertical_accel: 500 ft/min/s ≈ 30 m/s² ≈ 3g (выше структурного предела airliner ~2.5g sustained). Threshold 2000 ft/min/s в первоначальном плане был мёртвым (max в данных = 1126), 500 это реальный физический порог, поправлено по диагностическому анализу.
- energy_rate: 200 m/s, wide-diff artefact threshold. P99.9 = 49 m/s, значения > 200 практически всегда numerical artefacts на границах.

### Phase inconsistency

После диагностики `energy_deviation` выявлен phase classifier artefact: `smooth_phases` поглощает короткие сегменты в окружающую фазу, что иногда создаёт alt-phase mismatches.

- low-alt phases (ground/takeoff/landing) при altitude > 5000 ft: гарантированный artefact (raw rules требуют < 1500 / < 1500 / < 500 ft).
- cruise при altitude < 5000 ft: physical floor (raw cruise rule требует ≥ 15 000 ft, 5000 ft это безопасная нижняя граница, не ловит легитимные level-off на FL080).
- approach при altitude > 15 000 ft: raw approach rule требует alt < 10 000 ft, выше 15 000 это symmetric artefact.

### Energy deviation extreme

`|energy_deviation| > 10` помечает derived feature outliers. По диагностике 98% таких точек это phase classifier artefacts, но threshold имеет самостоятельную интерпретацию: точка вне 10x corridor width своей фазы.

In [3]:
GAP_THRESHOLD_SEC = 5.0

DQ_ACC_THRESHOLD = 15.0       # kts/s
DQ_TURN_THRESHOLD = 10.0      # deg/s
DQ_VACC_THRESHOLD = 500.0     # ft/min/s
DQ_ENERGY_RATE_THRESHOLD = 200.0  # m/s

# phase x altitude inconsistency thresholds
PHASE_LOW_ALT_PHASES = {0, 1, 6}  # ground / takeoff / landing
PHASE_LOW_ALT_CEILING = 5000  # ft, выше = artefact
PHASE_CRUISE_FLOOR = 5000     # ft, ниже = artefact (cruise = phase 3)
PHASE_APPROACH_CEILING = 15000  # ft, выше = artefact (approach = phase 5)

ENERGY_DEVIATION_EXTREME = 10.0

print('Thresholds:')
print(f'  gap_flag:                  dt_sec > {GAP_THRESHOLD_SEC}s')
print(f'  dq_acc_bad:                |acceleration| > {DQ_ACC_THRESHOLD} kts/s')
print(f'  dq_turn_bad:               |turn_rate| > {DQ_TURN_THRESHOLD} deg/s')
print(f'  dq_vacc_bad:               |vertical_accel| > {DQ_VACC_THRESHOLD} ft/min/s')
print(f'  dq_energy_rate_bad:        |energy_rate| > {DQ_ENERGY_RATE_THRESHOLD} m/s')
print(f'  phase_low_alt_artefact:    phase in (ground, takeoff, landing) and alt > {PHASE_LOW_ALT_CEILING} ft')
print(f'  phase_cruise_low_artefact: phase = cruise and alt < {PHASE_CRUISE_FLOOR} ft')
print(f'  phase_approach_high:       phase = approach and alt > {PHASE_APPROACH_CEILING} ft')
print(f'  energy_deviation_extreme:  |energy_deviation| > {ENERGY_DEVIATION_EXTREME}')

Thresholds:
  gap_flag:                  dt_sec > 5.0s
  dq_acc_bad:                |acceleration| > 15.0 kts/s
  dq_turn_bad:               |turn_rate| > 10.0 deg/s
  dq_vacc_bad:               |vertical_accel| > 500.0 ft/min/s
  dq_energy_rate_bad:        |energy_rate| > 200.0 m/s
  phase_low_alt_artefact:    phase ∈ {ground, takeoff, landing} & alt > 5000 ft
  phase_cruise_low_artefact: phase = cruise & alt < 5000 ft
  phase_approach_high:       phase = approach & alt > 15000 ft
  energy_deviation_extreme:  |energy_deviation| > 10.0


## 3. Discover input flights и chunk strategy

Pipeline: один проход по chunks по flight_id. В каждом chunk:
1. Считаем gap_flag (per-flight через flight boundaries).
2. Считаем derivative DQ flags.
3. Считаем phase inconsistency.
4. Считаем energy_deviation_extreme.
5. Объединяем в три contract'а.
6. Сохраняем chunk, потом стримим в финальный parquet.

Все операции vectorized numpy, не groupby.apply (на 3000 рейсах и ~5000 точек groupby.apply очень медленный).

In [4]:
# Берём все flight_id из enriched_v4
print('Discovering flight_ids...')
t0 = time.time()
all_fids_arr = pq.read_table(INPUT_PATH, columns=['flight_id']).column('flight_id').to_numpy()
all_fids_unique = np.unique(all_fids_arr)
n_total_flights = len(all_fids_unique)
del all_fids_arr
print(f'Total flights: {n_total_flights:,}  ({time.time() - t0:.0f}s)')

CHUNK_SIZE = 3000
flight_chunks = [all_fids_unique[i:i + CHUNK_SIZE] for i in range(0, n_total_flights, CHUNK_SIZE)]
print(f'Processing chunks: {len(flight_chunks)} (target {CHUNK_SIZE} flights each)')

del all_fids_unique

Discovering flight_ids...
Total flights: 29,788  (26s)
Processing chunks: 10 (target 3000 flights each)


0

## 4. Annotation function (vectorized per chunk)

Для каждого chunk DataFrame'а добавляет 8 новых колонок:
- `dt_sec`: float32, NaN для первой точки рейса.
- `gap_flag`: uint8, 1 если dt_sec > 5.
- `dq_derivative_bad`: uint8, OR из 4 sub-checks.
- `phase_inconsistent`: uint8, OR из 3 sub-checks.
- `energy_deviation_extreme`: uint8, |energy_deviation| > 10.
- `dq_hard_flag`: uint8, raw telemetry hard DQ.
- `dq_soft_flag`: uint8, suspicious derived dynamics.
- `feature_quality_flag`: uint8, derived feature artefacts.

In [5]:
def annotate_chunk(df):
    """Добавляет 8 DQ-аннотационных колонок к chunk DataFrame."""
    df = df.sort_values(['flight_id', 'timestamp']).reset_index(drop=True)
    n = len(df)

    # 1. dt_sec через flight boundaries
    fid = df['flight_id'].to_numpy()
    ts = df['timestamp'].astype('int64').to_numpy() / 1e9  # seconds since epoch

    # Разница с предыдущей точкой
    dt = np.empty(n, dtype=np.float32)
    dt[0] = np.nan
    dt[1:] = (ts[1:] - ts[:-1]).astype(np.float32)

    # NaN на границах рейсов (первая точка каждого нового рейса)
    flight_changed = np.empty(n, dtype=bool)
    flight_changed[0] = True
    flight_changed[1:] = fid[1:] != fid[:-1]
    dt[flight_changed] = np.nan

    df['dt_sec'] = dt

    # 2. gap_flag
    # NaN dt_sec (первая точка рейса) НЕ помечается gap, это просто начало
    gap = (dt > GAP_THRESHOLD_SEC) & np.isfinite(dt)
    df['gap_flag'] = gap.astype('uint8')

    # 3. dq_derivative_bad
    # Aggregate из 4 sub-checks (sub-flags не сохраняем, описано в markdown)
    acc = df['acceleration'].to_numpy()
    turn = df['turn_rate'].to_numpy()
    vacc = df['vertical_accel'].to_numpy()
    erate = df['energy_rate'].to_numpy()

    dq_acc = np.abs(acc) > DQ_ACC_THRESHOLD
    dq_turn = np.abs(turn) > DQ_TURN_THRESHOLD
    dq_vacc = np.abs(vacc) > DQ_VACC_THRESHOLD
    dq_erate = np.abs(erate) > DQ_ENERGY_RATE_THRESHOLD

    # NaN values становятся False, не помечаем NaN как bad (это другой issue)
    dq_acc = np.where(np.isfinite(acc), dq_acc, False)
    dq_turn = np.where(np.isfinite(turn), dq_turn, False)
    dq_vacc = np.where(np.isfinite(vacc), dq_vacc, False)
    dq_erate = np.where(np.isfinite(erate), dq_erate, False)

    dq_derivative = dq_acc | dq_turn | dq_vacc | dq_erate
    df['dq_derivative_bad'] = dq_derivative.astype('uint8')

    # 4. phase_inconsistent
    phase = df['flight_phase'].to_numpy()
    alt = df['altitude'].to_numpy()

    # Точки с NaN altitude не флагуем (они unknown phase или edge cases)
    alt_finite = np.isfinite(alt)

    low_alt_phase_mask = np.isin(phase, list(PHASE_LOW_ALT_PHASES))
    phase_low_artefact = low_alt_phase_mask & (alt > PHASE_LOW_ALT_CEILING) & alt_finite

    cruise_mask = phase == 3
    phase_cruise_artefact = cruise_mask & (alt < PHASE_CRUISE_FLOOR) & alt_finite

    approach_mask = phase == 5
    phase_approach_artefact = approach_mask & (alt > PHASE_APPROACH_CEILING) & alt_finite

    phase_inconsistent = phase_low_artefact | phase_cruise_artefact | phase_approach_artefact
    df['phase_inconsistent'] = phase_inconsistent.astype('uint8')

    # 5. energy_deviation_extreme
    edev = df['energy_deviation'].to_numpy()
    edev_extreme = np.abs(edev) > ENERGY_DEVIATION_EXTREME
    edev_extreme = np.where(np.isfinite(edev), edev_extreme, False)
    df['energy_deviation_extreme'] = edev_extreme.astype('uint8')

    # 6. dq_hard_flag (raw telemetry hard DQ)
    # threshold_bad по любому каналу
    threshold_any = (
        (df['altitude_threshold_bad'].to_numpy() == 1)
        | (df['groundspeed_threshold_bad'].to_numpy() == 1)
        | (df['vertical_rate_threshold_bad'].to_numpy() == 1)
    )
    stale = df['stale_altitude'].to_numpy() == 1

    dq_hard = threshold_any | stale | gap
    df['dq_hard_flag'] = dq_hard.astype('uint8')

    # 7. dq_soft_flag (suspicious derived dynamics)
    spike_any = (
        (df['altitude_spike_corrected'].to_numpy() == 1)
        | (df['groundspeed_spike_corrected'].to_numpy() == 1)
        | (df['vertical_rate_spike_corrected'].to_numpy() == 1)
    )

    dq_soft = spike_any | dq_derivative
    df['dq_soft_flag'] = dq_soft.astype('uint8')

    # 8. feature_quality_flag
    feature_quality = phase_inconsistent | edev_extreme
    df['feature_quality_flag'] = feature_quality.astype('uint8')

    return df


print('annotate_chunk defined.')

annotate_chunk defined.


## 5. Pass: annotate и save chunks

Один проход по chunks. Cleanup старых chunk-файлов перед стартом. Streaming через ParquetWriter в финальный файл будет в следующей секции.

In [6]:
print('Annotation pass: processing chunks...')

# Чистим stale-чанки от предыдущих прогонов
old_chunks = glob.glob(os.path.join(OUTPUT_DIR, 'annotated_v3_chunk_*.parquet'))
if old_chunks:
    for old in old_chunks:
        os.remove(old)
    print(f'Removed {len(old_chunks)} stale chunks from previous runs.')

t0 = time.time()
chunk_paths = []
total_rows = 0

for i, chunk_flights in enumerate(flight_chunks):
    df_chunk = pq.read_table(
        INPUT_PATH,
        filters=[('flight_id', 'in', set(chunk_flights))]
    ).to_pandas()
    df_chunk['timestamp'] = pd.to_datetime(df_chunk['timestamp'])

    # Annotate
    df_chunk = annotate_chunk(df_chunk)

    # Save chunk
    chunk_path = os.path.join(OUTPUT_DIR, f'annotated_v3_chunk_{i:02d}.parquet')
    df_chunk.to_parquet(chunk_path, index=False)
    chunk_paths.append(chunk_path)
    total_rows += len(df_chunk)

    print(f'  chunk {i+1:2d}/{len(flight_chunks)}: {len(df_chunk):,} rows to '
          f'{os.path.basename(chunk_path)}', end='\r')

    del df_chunk

print()
print(f'Annotation pass complete: {time.time() - t0:.0f}s')
print(f'Total rows: {total_rows:,}')
print(f'Chunks saved: {len(chunk_paths)}')
assert len(chunk_paths) == len(flight_chunks), (
    f'Expected {len(flight_chunks)} chunks, got {len(chunk_paths)}'
)

Annotation pass: processing chunks...
  chunk 10/10: 13,349,053 rows → annotated_v3_chunk_09.parquet
Annotation pass complete: 430s
Total rows: 149,129,454
Chunks saved: 10


## 6. Streaming merge в annotated_v3.parquet

Стримим chunks в финальный parquet через ParquetWriter, без удержания всего датасета в RAM (как в 02b_v4).

In [7]:
# Реконструируем chunk_paths через glob (на случай, если kernel перезапускался)
chunk_paths = sorted(glob.glob(os.path.join(OUTPUT_DIR, 'annotated_v3_chunk_*.parquet')))
print(f'Found {len(chunk_paths)} chunks on disk')
assert len(chunk_paths) > 0

print(f'Streaming merge to {OUTPUT_PATH}')
t0 = time.time()

# Открываем writer со schema из первого чанка
first_table = pq.read_table(chunk_paths[0])
schema = first_table.schema

total_rows = 0
unique_flights = set()

with pq.ParquetWriter(OUTPUT_PATH, schema, compression='snappy') as writer:
    for cp in chunk_paths:
        table = pq.read_table(cp)
        n_rows = table.num_rows
        total_rows += n_rows

        flights_in_chunk = set(table.column('flight_id').to_pylist())
        unique_flights.update(flights_in_chunk)

        writer.write_table(table)
        del table

        print(f'  {os.path.basename(cp)}: {n_rows:,} rows', end='\r')

print()
print(f'Merge complete: {time.time() - t0:.0f}s')
print(f'Total rows: {total_rows:,}')
print(f'Total flights: {len(unique_flights):,}')
print(f'File: {OUTPUT_PATH}')
print(f'Size: {os.path.getsize(OUTPUT_PATH) / 1e9:.2f} GB')

# Удаляем временные чанки
for cp in chunk_paths:
    if os.path.exists(cp):
        os.remove(cp)
print(f'Removed {len(chunk_paths)} temp chunks.')

Found 10 chunks on disk
Streaming merge → /content/drive/MyDrive/thesis_processed/european_flights_annotated_v3.parquet
  annotated_v3_chunk_09.parquet: 13,349,053 rows
Merge complete: 311s
Total rows: 149,129,454
Total flights: 29,788
File: /content/drive/MyDrive/thesis_processed/european_flights_annotated_v3.parquet
Size: 8.36 GB
Removed 10 temp chunks.


## 7. Quality check (disk-based)

Проверки на готовом parquet:
1. Schema: 41 колонка (33 enriched + 8 новых).
2. Counts per флаг.
3. Phase x dq_hard breakdown.
4. Phase x phase_inconsistent breakdown, особенно интересно для интерпретации.
5. dt_sec distribution (P50, P95, P99, P99.9).
6. Sanity assertion: phase_inconsistent ≥ 8485 (то, что было в diagnostic).
7. Intersection: phase_inconsistent AND dq_hard, должно быть низко (это разные категории).

In [8]:
print('Final Dataset Overview')
pf = pq.ParquetFile(OUTPUT_PATH)
total_rows = pf.metadata.num_rows
all_cols = pf.schema.names
print(f'Rows:    {total_rows:,}')
print(f'Columns ({len(all_cols)}):')
for c in all_cols:
    print(f'  {c}')

# Schema check
EXPECTED_NEW_COLS = [
    'dt_sec', 'gap_flag',
    'dq_derivative_bad', 'phase_inconsistent', 'energy_deviation_extreme',
    'dq_hard_flag', 'dq_soft_flag', 'feature_quality_flag',
]

actual_set = set(all_cols)
missing_new = [c for c in EXPECTED_NEW_COLS if c not in actual_set]
assert not missing_new

# 33 (enriched_v4) + 8 (new) = 41
assert len(all_cols) == 41
print(f'\nSchema check: 41 columns (33 enriched + 8 new), OK')

=== Final Dataset Overview ===
Rows:    149,129,454
Columns (41):
  flight_id
  timestamp
  latitude
  longitude
  icao24
  altitude
  groundspeed
  vertical_rate
  acceleration
  turn_rate
  vertical_accel
  wind_speed
  headwind
  crosswind
  energy_ratio
  energy_rate
  energy_deviation
  specific_energy
  kinetic_energy
  flight_phase
  phase_transition
  altitude_phase_z
  groundspeed_phase_z
  vertical_rate_phase_z
  specific_energy_phase_z
  energy_rate_phase_z
  altitude_threshold_bad
  groundspeed_threshold_bad
  vertical_rate_threshold_bad
  altitude_spike_corrected
  groundspeed_spike_corrected
  vertical_rate_spike_corrected
  stale_altitude
  dt_sec
  gap_flag
  dq_derivative_bad
  phase_inconsistent
  energy_deviation_extreme
  dq_hard_flag
  dq_soft_flag
  feature_quality_flag

Schema check: 41 columns (33 enriched + 8 new) ✓


In [9]:
def _load_col(col):
    return pq.read_table(OUTPUT_PATH, columns=[col]).column(col).to_numpy(zero_copy_only=False)

In [10]:
print('\n=== Counts per new flag ===')
flag_counts = {}
for flag in ['gap_flag', 'dq_derivative_bad', 'phase_inconsistent',
             'energy_deviation_extreme', 'dq_hard_flag', 'dq_soft_flag',
             'feature_quality_flag']:
    arr = _load_col(flag)
    n = int(arr.sum())
    pct = n / total_rows * 100
    flag_counts[flag] = n
    print(f'  {flag:>20s}: {n:,} ({pct:6.3f}%)')
    del arr

print('\n=== dt_sec distribution ===')
dt_arr = _load_col('dt_sec')
dt_finite = dt_arr[np.isfinite(dt_arr)]
print(f'  finite values: {len(dt_finite):,}')
for q in [0.5, 0.95, 0.99, 0.999, 0.9999, 1.0]:
    val = np.quantile(dt_finite, q)
    print(f'  P{q*100:>6.2f}: {val:>10.3f}s')

# Распределение для gap thresholds
print('\n  Gap counts at various thresholds:')
for thr in [2, 5, 10, 30, 60, 300]:
    n = int((dt_finite > thr).sum())
    pct = n / total_rows * 100
    print(f'    dt_sec > {thr:>3d}s: {n:,} ({pct:6.4f}%)')
del dt_arr, dt_finite

print('\n=== dq_hard breakdown by phase ===')
phase_arr = _load_col('flight_phase')
hard_arr = _load_col('dq_hard_flag')

for phase_code in range(-1, 7):
    phase_mask = phase_arr == phase_code
    phase_total = int(phase_mask.sum())
    if phase_total == 0:
        continue
    n_hard = int(hard_arr[phase_mask].sum())
    pct = n_hard / phase_total * 100
    print(f'  {PHASE_NAMES[phase_code]:>10s}: '
          f'{n_hard:,} / {phase_total:,} ({pct:6.3f}%)')
del hard_arr

print('\n=== phase_inconsistent breakdown by phase ===')
pi_arr = _load_col('phase_inconsistent')

for phase_code in range(-1, 7):
    phase_mask = phase_arr == phase_code
    phase_total = int(phase_mask.sum())
    if phase_total == 0:
        continue
    n_pi = int(pi_arr[phase_mask].sum())
    pct = n_pi / phase_total * 100
    if n_pi > 0:
        print(f'  {PHASE_NAMES[phase_code]:>10s}: '
              f'{n_pi:,} / {phase_total:,} ({pct:6.3f}%)')

# Diagnostic-confirmed expectation: ~8 651 точек с energy_deviation_extreme,
# из них ~8 485 phase_inconsistent. Проверяем как sanity.
n_pi_total = int(pi_arr.sum())
print(f'\n  Total phase_inconsistent: {n_pi_total:,}')
if n_pi_total < 8000:
    print('  WARNING: ожидали >= 8 000 (по диагностике 02b). Проверьте логику.')
elif n_pi_total > 50000:
    print('  WARNING: больше ожидаемого. Проверьте, не слишком ли широкие thresholds.')
else:
    print('  In expected range (>= 8 000 from diagnostic).')

del pi_arr, phase_arr


=== Counts per new flag ===
                      gap_flag:       72,111 ( 0.048%)
             dq_derivative_bad:      125,366 ( 0.084%)
            phase_inconsistent:       10,290 ( 0.007%)
      energy_deviation_extreme:        8,651 ( 0.006%)
                  dq_hard_flag:    1,889,024 ( 1.267%)
                  dq_soft_flag:      251,927 ( 0.169%)
          feature_quality_flag:       13,945 ( 0.009%)

=== dt_sec distribution ===
  finite values: 149,099,666
  P 50.00:      1.000s
  P 95.00:      1.000s
  P 99.00:      1.000s
  P 99.90:      2.000s
  P 99.99:     59.000s
  P100.00:  18390.000s

  Gap counts at various thresholds:
    dt_sec >   2s:    137,557 (0.0922%)
    dt_sec >   5s:     72,111 (0.0484%)
    dt_sec >  10s:     44,649 (0.0299%)
    dt_sec >  30s:     21,954 (0.0147%)
    dt_sec >  60s:     14,666 (0.0098%)
    dt_sec > 300s:      3,589 (0.0024%)

=== dq_hard breakdown by phase ===
     unknown:     60,748 /    667,689 ( 9.098%)
      ground:     14,253 /   

0

In [11]:
print('\n=== Intersection checks ===')
# phase_inconsistent AND dq_hard - должно быть НИЗКО (разные категории).
# По дизайну: phase_inconsistent - derived feature artefact, не raw telemetry.
hard = _load_col('dq_hard_flag')
pi = _load_col('phase_inconsistent')
soft = _load_col('dq_soft_flag')
fq = _load_col('feature_quality_flag')

inter_pi_hard = int((pi & hard).sum())
inter_pi_soft = int((pi & soft).sum())
inter_hard_soft = int((hard & soft).sum())
inter_hard_fq = int((hard & fq).sum())

print(f'  phase_inconsistent ∩ dq_hard:        {inter_pi_hard:,}')
print(f'  phase_inconsistent ∩ dq_soft:        {inter_pi_soft:,}')
print(f'  dq_hard ∩ dq_soft:                   {inter_hard_soft:,}')
print(f'  dq_hard ∩ feature_quality:           {inter_hard_fq:,}')

# Total dq_any (для общей оценки покрытия флагов)
any_flag = hard | soft | fq
n_any = int(any_flag.sum())
pct_any = n_any / total_rows * 100
print(f'\n  Any DQ/quality flag set:             {n_any:,} ({pct_any:.3f}%)')
print(f'  Clean points (no flags):             {total_rows - n_any:,} '
      f'({(1 - n_any/total_rows) * 100:.3f}%)')

del hard, pi, soft, fq, any_flag


=== Intersection checks ===
  phase_inconsistent ∩ dq_hard:             1,086
  phase_inconsistent ∩ dq_soft:             3,022
  dq_hard ∩ dq_soft:                       15,686
  dq_hard ∩ feature_quality:                1,113

  Any DQ/quality flag set:                2,134,791 (1.432%)
  Clean points (no flags):              146,994,663 (98.568%)


0

## 8. Top points by category для дашборда и интерпретации

Несколько примеров для каждой категории, помогает на защите объяснить, что ловит каждый флаг.

In [12]:
print('\n=== Examples per category ===')

# Загружаем компактный набор для examples
example_cols = [
    'flight_id', 'timestamp', 'altitude', 'groundspeed', 'vertical_rate',
    'flight_phase', 'energy_deviation', 'dt_sec',
    'dq_hard_flag', 'dq_soft_flag', 'feature_quality_flag',
    'gap_flag', 'phase_inconsistent', 'dq_derivative_bad',
]

print('\nLoading sample for examples (first 1M rows)...')
# Берём первые 1M строк, этого достаточно, чтобы найти примеры
df_sample = pq.read_table(
    OUTPUT_PATH,
    columns=example_cols,
).to_pandas().head(1_000_000)
df_sample['phase_name'] = df_sample['flight_phase'].map(PHASE_NAMES)

# Примеры для gap_flag
gap_examples = df_sample[df_sample['gap_flag'] == 1].head(5)
if len(gap_examples) > 0:
    print('\n  gap_flag examples (first 5):')
    for _, row in gap_examples.iterrows():
        print(f'    flight {int(row["flight_id"])}: dt_sec={row["dt_sec"]:.1f}s, '
              f'alt={row["altitude"]:.0f}, phase={row["phase_name"]}')

# Примеры для phase_inconsistent
pi_examples = df_sample[df_sample['phase_inconsistent'] == 1].head(5)
if len(pi_examples) > 0:
    print('\n  phase_inconsistent examples (first 5):')
    for _, row in pi_examples.iterrows():
        print(f'    flight {int(row["flight_id"])}: alt={row["altitude"]:.0f} ft, '
              f'phase={row["phase_name"]}, edev={row["energy_deviation"]:.1f}')

# Примеры для dq_derivative_bad
dd_examples = df_sample[df_sample['dq_derivative_bad'] == 1].head(5)
if len(dd_examples) > 0:
    print('\n  dq_derivative_bad examples (first 5):')
    for _, row in dd_examples.iterrows():
        print(f'    flight {int(row["flight_id"])}: alt={row["altitude"]:.0f}, '
              f'phase={row["phase_name"]}, edev={row["energy_deviation"]:.1f}')

del df_sample


=== Examples per category ===

Loading sample for examples (first 1M rows)...

  gap_flag examples (first 5):
    flight 256681304: dt_sec=262.0s, alt=34025, phase=cruise
    flight 256681305: dt_sec=22.0s, alt=33025, phase=cruise
    flight 256681390: dt_sec=138.0s, alt=325, phase=takeoff
    flight 256681486: dt_sec=7.0s, alt=32000, phase=cruise
    flight 256681494: dt_sec=18.0s, alt=37000, phase=cruise

  phase_inconsistent examples (first 5):
    flight 256682891: alt=700 ft, phase=cruise, edev=-6.9
    flight 256682990: alt=22400 ft, phase=landing, edev=69.3
    flight 256682990: alt=22400 ft, phase=landing, edev=69.3
    flight 256682990: alt=22400 ft, phase=landing, edev=69.2
    flight 256682990: alt=23000 ft, phase=landing, edev=71.1

  dq_derivative_bad examples (first 5):
    flight 256681304: alt=4175, phase=approach, edev=-0.0
    flight 256681304: alt=4175, phase=approach, edev=-0.0
    flight 256681390: alt=12925, phase=descent, edev=-0.8
    flight 256681462: alt=3075

0

## 9. DQ contract for 03_models

Финальное объявление контракта между 02c и 03_models. После этой ячейки 03_models должен использовать именно эти константы и комбинации флагов для выбора training pool, scoring all, dashboard categories.

In [13]:
# Все DQ-related колонки в annotated_v3
DQ_RAW_FLAGS = DQ_FLAGS_PREPROCESS  # 7 флагов из 02_preprocessing_v3

DQ_NEW_FLAGS = [
    'gap_flag',
    'dq_derivative_bad',
    'phase_inconsistent',
    'energy_deviation_extreme',
]

DQ_CONTRACTS = [
    'dq_hard_flag',       # raw telemetry hard DQ
    'dq_soft_flag',       # suspicious derived dynamics
    'feature_quality_flag',  # derived feature / phase-classifier artefacts
]

DT_COLS = ['dt_sec']

# Итого: 7 raw + 4 new sub-flags + 3 contracts + 1 dt_sec = 15 DQ-related cols
print('\nDQ contract for 03_models:')
print(f'\n  Raw flags from 02_preprocessing ({len(DQ_RAW_FLAGS)}):')
for f in DQ_RAW_FLAGS:
    print(f'    {f}')
print(f'\n  New sub-flags from 02c ({len(DQ_NEW_FLAGS)}):')
for f in DQ_NEW_FLAGS:
    print(f'    {f}')
print(f'\n  Combined contracts ({len(DQ_CONTRACTS)}):')
for f in DQ_CONTRACTS:
    print(f'    {f}')
print(f'\n  Temporal helper ({len(DT_COLS)}):')
for f in DT_COLS:
    print(f'    {f}')

# Проверяем, что все ожидаемые колонки на месте
all_expected = set(DQ_RAW_FLAGS + DQ_NEW_FLAGS + DQ_CONTRACTS + DT_COLS)
all_actual = set(pq.ParquetFile(OUTPUT_PATH).schema.names)
missing = all_expected - all_actual
assert not missing
print('\nAll DQ contract columns present.')

# Рекомендуемое использование в 03_models:
print('\nRecommended usage in 03_models:')
print('  LSTM-AE clean training pool:')
print('    df.query("dq_hard_flag == 0 AND gap_flag == 0")')
print('    plus исключение windows с любым dq_hard в +-5 точек (делается в 03)')
print('  IF / HDBSCAN training pool:')
print('    df.query("dq_hard_flag == 0")')
print('  Scoring (все три модели):')
print('    score everything, dashboard категоризует по dq_hard / dq_soft / feature_quality')

print(f'\nFinal: {OUTPUT_PATH}')
print(f'Rows: {total_rows:,}, Flights: {len(unique_flights)}')
print(f'Columns: 41, Size: {os.path.getsize(OUTPUT_PATH) / 1e9:.2f} GB')
print('02c_v3 complete. Ready for 03_models.')


DQ contract for 03_models:

  Raw flags from 02_preprocessing (7):
    altitude_threshold_bad
    groundspeed_threshold_bad
    vertical_rate_threshold_bad
    altitude_spike_corrected
    groundspeed_spike_corrected
    vertical_rate_spike_corrected
    stale_altitude

  New sub-flags from 02c (4):
    gap_flag
    dq_derivative_bad
    phase_inconsistent
    energy_deviation_extreme

  Combined contracts (3):
    dq_hard_flag
    dq_soft_flag
    feature_quality_flag

  Temporal helper (1):
    dt_sec

All DQ contract columns present. ✓

Recommended usage in 03_models:
  LSTM-AE clean training pool:
    df.query("dq_hard_flag == 0 AND gap_flag == 0")
    + exclude windows containing any dq_hard within ±5 points (built in 03)
  IF / HDBSCAN training pool:
    df.query("dq_hard_flag == 0")
  Scoring (все три модели):
    score everything; dashboard categorizes by dq_hard / dq_soft / feature_quality

Final: /content/drive/MyDrive/thesis_processed/european_flights_annotated_v3.parquet
R